# Train CNN


In [1]:
import json
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


## Set input and output paths


In [2]:
run_dir = Path('/u/akazantsev/ml_dl_for_rfi_mitigation/runs/punch_rfi_reproduction')
base_name = 'B0531+21_59000_48386'

channels_path = run_dir / f'{base_name}_training_channels.npy'
metadata_path = run_dir / f'{base_name}_training_metadata.csv'
split_path = run_dir / f'{base_name}_training_split_indices.npz'
checkpoint_path = run_dir / 'best_model_by_acc.pt'
history_path = run_dir / 'cnn_training_history.csv'
training_summary_path = run_dir / 'cnn_training_summary.json'
replace_existing_outputs = True

for input_path in (channels_path, metadata_path, split_path):
    if not input_path.is_file():
        raise FileNotFoundError(f'Required input does not exist: {input_path}')

# Use this only for a complete rerun of the CNN evaluation path.
existing_outputs = [
    output_path for output_path in (checkpoint_path, history_path, training_summary_path)
    if output_path.exists()
]
if existing_outputs and not replace_existing_outputs:
    raise FileExistsError(f'CNN output already exists: {existing_outputs[0]}')

random_state = 42
np.random.seed(random_state)
torch.manual_seed(random_state)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(random_state)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


## Load and check training data


In [3]:
metadata = pd.read_csv(metadata_path)
required_columns = {'sample_index', 'label'}
missing_columns = required_columns.difference(metadata.columns)
if missing_columns:
    raise ValueError(f'Metadata columns are missing: {sorted(missing_columns)}')

training_channels = np.load(channels_path)
if training_channels.shape != (len(metadata), 256):
    raise ValueError('Training channels must have shape (metadata_rows, 256)')
if training_channels.dtype != np.float32:
    raise ValueError('Training channels must use float32 values')
if not np.isfinite(training_channels).all():
    raise ValueError('Training channels contain non-finite values')
if not np.array_equal(metadata['sample_index'].to_numpy(), np.arange(len(metadata))):
    raise ValueError('Metadata rows do not match channel rows')

label_names = metadata['label'].to_numpy()
if set(np.unique(label_names)) != {'BGN', 'NBRFI'}:
    raise ValueError('Training metadata must contain only BGN and NBRFI labels')

labels = (label_names == 'NBRFI').astype(np.float32)
class_rows = np.bincount(labels.astype(np.int64), minlength=2)
if not np.array_equal(class_rows, np.array([10_000, 10_000])):
    raise ValueError(f'Expected 10,000 rows per class, got {class_rows.tolist()}')

# Preserve the reported per-channel maximum normalization.
channel_maxima = np.max(np.abs(training_channels), axis=1, keepdims=True)
channel_maxima[channel_maxima < 1e-8] = 1.0
training_channels = training_channels / channel_maxima

with np.load(split_path) as split_data:
    expected_split_names = {'train_indices', 'validation_indices', 'test_indices'}
    if set(split_data.files) != expected_split_names:
        raise ValueError(f'Split file must contain {sorted(expected_split_names)}')
    train_indices = split_data['train_indices'].astype(np.int64)
    validation_indices = split_data['validation_indices'].astype(np.int64)
    test_indices = split_data['test_indices'].astype(np.int64)

all_split_indices = np.concatenate((train_indices, validation_indices, test_indices))
if all_split_indices.min() < 0 or all_split_indices.max() >= len(metadata):
    raise ValueError('Split indices are outside training metadata')
if len(np.unique(all_split_indices)) != len(metadata):
    raise ValueError('Split indices must cover every training row exactly once')
if (len(train_indices), len(validation_indices), len(test_indices)) != (16_000, 2_000, 2_000):
    raise ValueError('Split sizes are invalid')

for split_name, split_indices in (
    ('train', train_indices),
    ('validation', validation_indices),
    ('test', test_indices),
):
    if labels[split_indices].mean() != 0.5:
        raise ValueError(f'{split_name} split is not balanced')

print('Training channels:', training_channels.shape)
print('Class rows [BGN, NBRFI]:', class_rows.tolist())
print('Split rows:', len(train_indices), len(validation_indices), len(test_indices))


Training channels: (20000, 256)
Class rows [BGN, NBRFI]: [10000, 10000]
Split rows: 16000 2000 2000


## Create datasets and loaders


In [4]:
class FrequencyChannelDataset(Dataset):
    """Return one normalized 256-sample channel and its binary label."""

    def __init__(self, channel_time_series, labels, sample_indices):
        self.channel_time_series = channel_time_series
        self.labels = labels
        self.sample_indices = sample_indices

    def __len__(self):
        return len(self.sample_indices)

    def __getitem__(self, item_index):
        sample_index = self.sample_indices[item_index]
        channel_time_series = np.asarray(
            self.channel_time_series[sample_index], dtype=np.float32
        )
        if channel_time_series.shape != (256,):
            raise ValueError('Each channel time series must contain 256 samples')

        return (
            torch.from_numpy(channel_time_series).unsqueeze(0),
            torch.tensor(self.labels[sample_index], dtype=torch.float32),
        )


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size = 256

train_dataset = FrequencyChannelDataset(training_channels, labels, train_indices)
validation_dataset = FrequencyChannelDataset(
    training_channels, labels, validation_indices
)

loader_options = {
    'batch_size': batch_size,
    'num_workers': 0,
    'pin_memory': device.type == 'cuda'
}
train_loader = DataLoader(
    train_dataset,
    shuffle=True,
    generator=torch.Generator().manual_seed(random_state),
    **loader_options
)
validation_loader = DataLoader(validation_dataset, shuffle=False, **loader_options)

first_batch, _ = next(iter(train_loader))
if first_batch.shape != (batch_size, 1, 256) or not torch.isfinite(first_batch).all():
    raise ValueError('Training batch is invalid')

if device.type == 'cuda':
    print('Training device: GPU', torch.cuda.get_device_name(device))
else:
    print('Training device: CPU')
print('Training batch shape:', tuple(first_batch.shape))


Training device: GPU Quadro RTX 6000
Training batch shape: (256, 1, 256)


## Define CNN


In [5]:
class CNN1D_RFI_256_logits(nn.Module):
    """Return one RFI logit for one 256-sample channel."""

    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 64, kernel_size=7)
        self.pool1 = nn.MaxPool1d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=7)
        self.pool2 = nn.MaxPool1d(kernel_size=2, stride=2)
        self.conv3 = nn.Conv1d(128, 256, kernel_size=10)
        self.pool3 = nn.MaxPool1d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(p=0.5)
        self.fc1 = nn.Linear(6400, 256)
        self.fc2 = nn.Linear(256, 1)

    def forward(self, inputs):
        outputs = self.pool1(F.relu(self.conv1(inputs)))
        outputs = F.relu(self.conv2(outputs))
        # Preserve the reported 6,400-input dense layer.
        outputs = F.pad(outputs, (0, 1))
        outputs = self.pool2(outputs)
        outputs = self.pool3(F.relu(self.conv3(outputs)))
        outputs = self.flatten(outputs)
        outputs = self.dropout(F.relu(self.fc1(outputs)))
        return self.fc2(outputs)


model = CNN1D_RFI_256_logits().to(device)
if model(torch.zeros(1, 1, 256, device=device)).shape != (1, 1):
    raise ValueError('CNN output shape is invalid')

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print('Trainable parameters:', sum(parameter.numel() for parameter in model.parameters()))


Trainable parameters: 2024833


## Train CNN


In [6]:
def run_one_epoch(data_loader, training):
    """Run one epoch and return mean loss and accuracy."""

    model.train(training)
    total_loss = 0.0
    correct_rows = 0
    total_rows = 0

    for batch_inputs, batch_labels in data_loader:
        batch_inputs = batch_inputs.to(device, non_blocking=True)
        batch_labels = batch_labels.to(device, non_blocking=True).unsqueeze(1)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            logits = model(batch_inputs)
            loss = criterion(logits, batch_labels)

            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * len(batch_labels)
        predictions = (torch.sigmoid(logits) >= 0.5).to(torch.int64)
        correct_rows += (predictions == batch_labels.to(torch.int64)).sum().item()
        total_rows += len(batch_labels)

    return total_loss / total_rows, correct_rows / total_rows


n_epochs = 50
best_validation_accuracy = -np.inf
history_rows = []
if device.type == 'cuda':
    torch.cuda.synchronize(device)
training_started = perf_counter()
epoch_progress = tqdm(range(1, n_epochs + 1), desc='Training', unit='epoch')

for epoch_number in epoch_progress:
    train_loss, train_accuracy = run_one_epoch(train_loader, training=True)
    validation_loss, validation_accuracy = run_one_epoch(
        validation_loader, training=False
    )

    history_rows.append({
        'epoch': epoch_number,
        'train_loss': train_loss,
        'train_accuracy': train_accuracy,
        'validation_loss': validation_loss,
        'validation_accuracy': validation_accuracy
    })

    # Do not use test rows during training or model selection.
    if validation_accuracy > best_validation_accuracy:
        best_validation_accuracy = validation_accuracy
        torch.save(
            {
                'epoch': epoch_number,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'validation_loss': validation_loss,
                'validation_accuracy': validation_accuracy,
            },
            checkpoint_path,
        )

    epoch_progress.set_postfix(
        train_loss=f'{train_loss:.6f}',
        train_accuracy=f'{train_accuracy:.4f}',
        validation_loss=f'{validation_loss:.6f}',
        validation_accuracy=f'{validation_accuracy:.4f}',
    )

if device.type == 'cuda':
    torch.cuda.synchronize(device)
training_wall_clock_s = perf_counter() - training_started
training_history = pd.DataFrame(history_rows)
if len(training_history) != n_epochs:
    raise RuntimeError('CNN training history is incomplete')
training_history.to_csv(history_path, index=False)


Training:   0%|          | 0/50 [00:00<?, ?epoch/s]

## Save best model


In [7]:
if not checkpoint_path.is_file():
    raise RuntimeError('Best validation checkpoint was not saved')

checkpoint = torch.load(checkpoint_path, map_location=device)
required_checkpoint_keys = {
    'epoch',
    'model_state_dict',
    'optimizer_state_dict',
    'validation_loss',
    'validation_accuracy',
}
if set(checkpoint) != required_checkpoint_keys:
    raise RuntimeError('Checkpoint has an unexpected schema')

print('Best checkpoint epoch:', checkpoint['epoch'])
print('Best validation loss:', checkpoint['validation_loss'])
print('Best validation accuracy:', checkpoint['validation_accuracy'])
training_summary = {
    'device': device.type,
    'n_epochs': n_epochs,
    'random_state': random_state,
    'training_wall_clock_s': training_wall_clock_s,
    'timer_scope': 'full CNN training loop with validation after each epoch',
    'best_checkpoint_epoch': int(checkpoint['epoch']),
    'best_validation_loss': float(checkpoint['validation_loss']),
    'best_validation_accuracy': float(checkpoint['validation_accuracy'])
}
with training_summary_path.open('w', encoding='utf-8') as handle:
    json.dump(training_summary, handle, indent=2, sort_keys=True)
    handle.write('\n')

print('Checkpoint:', checkpoint_path)
print('Training history:', history_path)
print('Training summary:', training_summary_path)
print('Training time (s):', training_wall_clock_s)


Best checkpoint epoch: 19
Best validation loss: 0.20245591259002685
Best validation accuracy: 0.925
Checkpoint: /u/akazantsev/ml_dl_for_rfi_mitigation/runs/punch_rfi_reproduction/best_model_by_acc.pt
Training history: /u/akazantsev/ml_dl_for_rfi_mitigation/runs/punch_rfi_reproduction/cnn_training_history.csv
Training summary: /u/akazantsev/ml_dl_for_rfi_mitigation/runs/punch_rfi_reproduction/cnn_training_summary.json
Training time (s): 243.51165959471837
